# 01 - Data quality and data realism

**Question:** Can the numbers in this project be trusted, and what kind of data is this?

**Method:** (1) run the data-quality gates on the frozen cleaned dataset, (2) list what is kept but excluded from specific
analyses, (3) measure how far the data departs from what real operational logs look like, (4) recompute the headline KPIs
independently (standard-library Python, no pandas) and compare.

Everything below reads the same controlled analytical dataset the rest of the project uses.


In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "routeiq").exists())
sys.path.insert(0, str(ROOT / "src"))
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 110)
from routeiq.analysis import figures
figures.apply_theme()
from routeiq.config import FIGURES_DIR, TABLES_DIR, CLEANED_DELIVERY_CSV, RAW_DATA_CSV
from routeiq.features.analytical import load_analytical_dataset
df = load_analytical_dataset()
tbl = lambda name: pd.read_csv(TABLES_DIR / f"{name}.csv")
show = lambda name: display(Image(filename=str(FIGURES_DIR / name)))
say = lambda s: display(Markdown(s))
print(f"{len(df):,} deliveries loaded from the controlled analytical dataset")


43,648 deliveries loaded from the controlled analytical dataset


## 1. Data-quality gates


In [2]:
from routeiq.validation.data_quality import data_quality_checks, exclusion_summary
cleaned = pd.read_csv(CLEANED_DELIVERY_CSV)
dq = data_quality_checks(cleaned)
display(dq)
assert (dq["status"] == "PASS").all()
say(f"**{(dq.status == 'PASS').sum()} of {len(dq)} checks pass.** These are the same rules `pytest` enforces on every run.")


,check,expected,actual,status
0,row count,43648,43648,PASS
1,duplicate Order_ID,0,0,PASS
2,fully duplicated records,0,0,PASS
3,"null key columns (Order_ID, date, category, area, traffic, weather, vehicle, delivery time)",0,0,PASS
4,Traffic: values outside the documented set,0,0,PASS
5,Weather: values outside the documented set,0,0,PASS
6,Area: values outside the documented set,0,0,PASS
7,Vehicle: values outside the documented set,0,0,PASS
8,categories,16,16,PASS
9,delivery time <= 0,0,0,PASS


**19 of 19 checks pass.** These are the same rules `pytest` enforces on every run.

## 2. Rows kept but excluded from specific analyses
Nothing is silently dropped: each group below stays in the dataset and is excluded only where its use would mislead.


In [3]:
display(exclusion_summary(cleaned))


,group,rows,excluded_from,share_of_dataset_pct
0,No valid agent rating,54,rating analysis and driver model,0.124
1,Invalid coordinates (distance unavailable),3651,distance analysis,8.365
2,Area = Other (not a formal tier),1136,tier comparisons (kept elsewhere),2.603
3,"Area = Semi-Urban (n=152, 100% breach)",152,driver model (complete separation); reported descriptively,0.348


## 3. Data realism: is this real operational data?

Several patterns are hard to explain as organic delivery operations. The table quantifies them. **This does not prove how the data
was made** (provenance cannot be verified from the repository); it measures how far the data departs from real operational logs,
so every later finding can be worded honestly.


In [4]:
from routeiq.analysis.data_realism import realism_checks
raw = pd.read_csv(RAW_DATA_CSV) if RAW_DATA_CSV.exists() else None
rc = realism_checks(df, raw)
display(rc[["check", "observed", "expected_if_real_operational_data", "evidence_of_rule_based_construction"]])


,check,observed,expected_if_real_operational_data,evidence_of_rule_based_construction
0,Delivery times that are multiples of 5 minutes,95.0% (89 distinct values),about 20% (timestamps measured to the minute),strong
1,Preparation time values,"only [5, 10, 15]; shares 33.5%, 33.2%, 33.3% (uniform-fit p=0.76)","a continuous, right-skewed distribution",strong
2,Traffic level determined by order hour,97.5% of deliveries match their hour's most common traffic level; Jam only in hours 19-22; 4 of 17 hours h...,"traffic varies within an hour (incidents, routes, weekdays)",strong
3,Hourly order volume,"no orders 01-07h; evening plateau 17-23h CV=0.03 (min 4247, max 4663)",smooth demand curve with peaks,moderate
4,Breach rate step at agent age 30,14.1% (age<30) vs 33.1% (age>=30); 29->30 jump 17.5 pts vs year-to-year SD 0.4 pts,"gradual change with age, not a cliff at a round number",strong
5,Breach rate step at agent rating 4.5,"4.4 -> 63.2%, 4.5 -> 10.5% (drop 52.7 pts)",gradual change with rating,strong
6,Weather effects collapse into three tiers,Sandstorms/Stormy/Windy coefficients differ by only 0.4 min; Fog vs Cloudy +0.2 min; Sunny vs the mid tier...,each weather type has its own effect,strong
7,A handful of additive rules explain delivery time,"OLS R2 = 0.62 using traffic, weather, area, vehicle, rating<4.5, age>=30, grocery",much lower explanatory power from coarse categories alone,moderate
8,Semi-Urban area,"n=152, 100% breach, 83% in Jam traffic, delivery time 245 min median",some variation; a perfect 100% is rare,strong
9,Raw file formatting and range defects,53 ratings of 6.0 (scale is 1-5); 3505 zero coordinates; 96% of Traffic/Vehicle/Area values padded with tr...,"some defects, but rarely this systematic",weak


In [5]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
res = (df["delivery_time_minutes"] % 5).value_counts(normalize=True).sort_index()
ax[0].bar(res.index.astype(str), 100 * res.values, color=figures.NAVY)
ax[0].set_title("Delivery time modulo 5"); ax[0].set_xlabel("minutes mod 5"); ax[0].set_ylabel("% of deliveries")
mix = df.groupby(["order_hour", "traffic"]).size().unstack(fill_value=0)
mix = mix.div(mix.sum(axis=1), axis=0)[["Low", "Medium", "High", "Jam"]]
mix.plot(kind="bar", stacked=True, ax=ax[1], color=["#B7C9D8", "#5E84A0", "#123B5D", figures.CORAL], width=0.85)
ax[1].set_title("Traffic level by order hour"); ax[1].set_ylabel("share of deliveries"); ax[1].legend(fontsize=8)
prep = df["prep_time_minutes"].value_counts(normalize=True).sort_index()
ax[2].bar(prep.index.astype(str), 100 * prep.values, color=figures.TEAL)
ax[2].set_title("Preparation time values"); ax[2].set_xlabel("minutes"); ax[2].set_ylim(0, 50)
plt.tight_layout(); plt.show()


### Reading
* 95% of delivery times are multiples of 5 minutes, and prep time takes only the values 5/10/15 in equal thirds.
* Traffic is almost a function of the hour (Jam only 19:00-22:00; Low in the morning). **Traffic and time of day are nearly the same variable here.**
* Breach rate steps at exactly age 30 and rating 4.5 (see notebook 03); weather effects collapse into three identical tiers.

**Framing used everywhere in this project:** *a realistic synthetic-style delivery dataset*. Findings describe patterns
in this dataset. They are not evidence about any real company's operations.


## 4. Independent recomputation of the headline KPIs


In [6]:
from routeiq.validation.independent import compute
ind = compute(CLEANED_DELIVERY_CSV)
k = df
cmp = pd.DataFrame({
    "independent (stdlib csv, own percentile)": [ind["total_deliveries"], ind["breached_deliveries"], ind["breach_rate_pct"], ind["avg_delivery_minutes"], ind["p90_delivery_minutes"]],
    "pandas analytical dataset": [len(k), int(k.breach_flag.sum()), 100 * k.breach_flag.mean(), k.delivery_time_minutes.mean(), np.percentile(k.delivery_time_minutes, 90)],
}, index=["total deliveries", "breached deliveries", "breach rate %", "average delivery minutes", "P90 delivery minutes"])
cmp["match"] = np.isclose(cmp.iloc[:, 0], cmp.iloc[:, 1])
display(cmp)
say(f"Stored SLA flag vs breach recomputed from scratch: **{ind['stored_flag_mismatches_vs_recomputed']} mismatches** out of {ind['total_deliveries']:,}.")
rec = tbl("reconciliation")
say(f"Full four-layer reconciliation (independent CSV, pandas outputs, SQL, v1 Power BI): **{(rec.status == 'PASS').sum()} of {len(rec)} metrics agree**; "
    f"exceptions: {', '.join(rec.loc[rec.status != 'PASS', 'metric'])} (the v1 Pareto tie label).")


,"independent (stdlib csv, own percentile)",pandas analytical dataset,match
total deliveries,"43,648.000","43,648.000",True
breached deliveries,"10,328.000","10,328.000",True
breach rate %,23.662,23.662,True
average delivery minutes,124.914,124.914,True
P90 delivery minutes,195.000,195.000,True


Stored SLA flag vs breach recomputed from scratch: **0 mismatches** out of 43,648.

Full four-layer reconciliation (independent CSV, pandas outputs, SQL, v1 Power BI): **121 of 122 metrics agree**; exceptions: pareto_cumulative_pct_rank_7 (the v1 Pareto tie label).

**What this reconciliation is, and is not.** Independent code paths reproduce the same numbers from the same file. That
catches implementation errors in any one layer. It cannot detect a fault in the source data, and it is not proof of
correctness.
